# Gradient Boosting

Last time we saw that random forests reduce variance by averaging many independent trees. Each tree is trained on a different bootstrap sample, and their errors cancel out through averaging. The key insight was that bagging works best with high-variance models - it smooths out noise but doesn't help if the base model is already too simple.

Boosting takes the opposite approach. Instead of training independent trees and averaging them, it builds trees *sequentially* - each new tree focuses specifically on the errors the previous ones made. Where bagging reduces variance, boosting reduces bias. A random forest can't fix a tree that's systematically wrong about certain regions of the feature space. Boosting can, because each new tree is aimed directly at the remaining mistakes.

This is the payoff lecture. The baseline hierarchy we've been building all semester - Dummy, Ridge, KNN, Decision Tree, Random Forest - culminates here. Gradient boosted trees, and XGBoost in particular, are arguably the single most important algorithm for tabular data in practice. They dominate Kaggle competitions, power production systems at scale, and are the default first choice for most structured data problems.

## Setup

In [ ]:
%matplotlib inline

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

In [ ]:
# Same dataset and split as 12b and 13a
california = fetch_california_housing()
X, y = california.data, california.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Test:     {X_test.shape[0]:,} samples")

Here's where we left off in 13a - the baseline hierarchy through Random Forest:

In [ ]:
# Recap from 13a: baseline hierarchy scores
recap_models = {
    "Dummy (mean)": DummyRegressor(),
    "Ridge": Ridge(),
    "KNN (scaled)": Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor())]),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
}

recap_rows = []
for name, model in recap_models.items():
    cv = cross_val_score(
        model, X_train, y_train, cv=5,
        scoring="neg_root_mean_squared_error",
    )
    recap_rows.append({"Model": name, "CV RMSE": f"{-cv.mean():.4f}"})

recap_df = pd.DataFrame(recap_rows)
print(recap_df.to_string(index=False))

The big jump was DT → RF: variance reduction through averaging. Today we ask: can we do better by attacking bias instead?

---

## Part 1: The Boosting Idea

### Learning from mistakes

Imagine you're predicting house prices and your first model guesses \\$200k for a house that's actually worth \\$300k. The residual - the error - is \\$100k. Now train a second model, not on the original prices, but on those residuals. If the second model predicts \\$30k for that same house, your combined prediction becomes \\$200k + \\$30k = \\$230k. The residual is now \\$70k instead of \\$100k. Train a third model on *those* residuals, and the error shrinks further. Repeat 100 or 500 times.

That's gradient boosting in a nutshell: fit a model, compute what it got wrong, fit the next model on the errors, and accumulate predictions. Each tree in the sequence is shallow (typically 3-5 levels) and individually weak - it captures just a little bit of the remaining pattern. The power comes from the accumulation.

Why "gradient"? When we fit on residuals, we're implicitly following the negative gradient of the squared-error loss function:

> The negative gradient of $(y - \hat{y})^2 / 2$ **is** the residual $y - \hat{y}$.

Let that sink in for a minute.

This is powerful because the same framework generalizes to any differentiable loss - absolute error, Huber loss, log-loss for classification - just by swapping the gradient. The "gradient" in gradient boosting means we're doing gradient descent in function space, one tree at a time.

![All roads lead through Gradient Descent](./images/13b-trees-boosting-gd.png)

This is fundamentally different from random forests. An RF trains 200 full-depth trees *independently* and averages them - it never looks at what other trees got wrong. A gradient boosted model trains 200 shallow trees *sequentially*, where each tree is specifically designed to correct the mistakes of everything that came before it.

### Why shallow trees?

In a random forest, each tree is grown deep to be individually as accurate as possible. Averaging takes care of the overfitting. In boosting, the strategy is different: each tree should make only a small correction. Deep trees would over-correct - they'd fit the residuals too closely and amplify noise rather than reduce it. Shallow trees (depth 3-5) act as "weak learners" that each nudge the prediction in the right direction without overshooting.

This is a general principle: bagging wants strong, diverse base learners. Boosting wants weak, cautious ones.

### The learning rate

The full residual correction from each tree isn't used directly. Instead, a *learning rate* (also called the shrinkage parameter) scales down each tree's contribution. If the learning rate is 0.1, only 10% of each tree's correction is added to the running prediction. This forces the model to take many small steps rather than a few large ones, which leads to better generalization.

If this sounds familiar, it should - it's the same concept as the step size in gradient descent from the regularization lecture (06a). Smaller steps converge more slowly but find better solutions. Larger steps are faster but risk overshooting. The learning rate and number of estimators trade off directly: a lower learning rate needs more trees to reach the same level of performance, but typically generalizes better.

In [ ]:
# Quick demonstration: boosting with 3 shallow trees
from sklearn.tree import DecisionTreeRegressor as DTR

# Use one feature for visualization
feat_idx = list(california.feature_names).index("MedInc")
X_1d_train = X_train[:, feat_idx:feat_idx+1]
X_1d_test = X_test[:, feat_idx:feat_idx+1]

# Sort for clean plotting
sort_idx = X_1d_test[:, 0].argsort()
X_1d_plot = X_1d_test[sort_idx]
y_plot = y_test[sort_idx]

# Manual boosting: 3 rounds, learning_rate = 0.5
lr = 0.5
residuals = y_train.copy()
predictions = np.zeros(len(X_1d_plot))
trees = []

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for i, ax in enumerate(axes):
    tree = DTR(max_depth=3, random_state=42)
    tree.fit(X_1d_train, residuals)
    trees.append(tree)

    correction = tree.predict(X_1d_plot)
    predictions += lr * correction

    # Update residuals for next round
    residuals = residuals - lr * tree.predict(X_1d_train)

    ax.scatter(X_1d_plot, y_plot, alpha=0.05, s=5, color="gray")
    ax.plot(X_1d_plot, predictions, color="tomato", linewidth=2)
    ax.set_xlabel("MedInc")
    ax.set_title(f"After {i+1} tree{'s' if i > 0 else ''}")
    if i == 0:
        ax.set_ylabel("House Value (\\$100k)")

plt.suptitle("Gradient Boosting: Sequential Residual Fitting", y=1.02)
plt.tight_layout()
plt.show()

Each panel adds one more tree's contribution. The first tree captures the broad trend - house values rise with median income. The second and third trees refine the shape, adding curvature that the first tree missed. With 100 or 500 trees (instead of 3), the approximation becomes very precise.

---

## Part 2: sklearn's GradientBoostingRegressor

SKL's basic implementation of Boosting is the `GradientBoostingRegressor` or the corresponding classifier. We'll use it here for a baseline of performance and time.

### Fitting a gradient boosted model

Let's use the full California Housing dataset with all 8 features. The key hyperparameters to set:

- `n_estimators`: number of sequential trees (more = more complex, risk of overfitting)
- `learning_rate`: shrinkage per tree (smaller = more conservative, needs more trees)
- `max_depth`: depth of each tree (typically 3-5 for boosting, much shallower than RF)
- `subsample`: fraction of training data used per tree (< 1.0 adds randomness, called *stochastic* gradient boosting)

In [ ]:
gb = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    random_state=42,
)
gb.fit(X_train, y_train)

gb_train_rmse = np.sqrt(mean_squared_error(y_train, gb.predict(X_train)))
gb_test_rmse = np.sqrt(mean_squared_error(y_test, gb.predict(X_test)))

print(f"Gradient Boosting (500 trees, lr=0.1, depth=4):")
print(f"  Train RMSE: {gb_train_rmse:.4f}")
print(f"  Test RMSE:  {gb_test_rmse:.4f}")

That test RMSE is a meaningful drop from RF's ~0.52 - roughly a 10% reduction! RF attacked variance by averaging; boosting is now attacking bias by sequentially correcting errors. These are complementary improvements, not redundant ones.

### The learning curve: when to stop

A unique feature of gradient boosting is `staged_predict` - it gives you the prediction at every stage of the boosting process (after tree 1, after tree 2, ..., after tree 500). This lets you see exactly when the model starts overfitting. With random forests, more trees always helps or at least doesn't hurt. With boosting, there's a sweet spot - too many trees and the model starts memorizing training noise.

In [ ]:
# Compute RMSE at each boosting stage
train_rmse_by_stage = []
test_rmse_by_stage = []

for y_pred in gb.staged_predict(X_train):
    train_rmse_by_stage.append(np.sqrt(mean_squared_error(y_train, y_pred)))

for y_pred in gb.staged_predict(X_test):
    test_rmse_by_stage.append(np.sqrt(mean_squared_error(y_test, y_pred)))

stages = np.arange(1, len(train_rmse_by_stage) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(stages, train_rmse_by_stage, label="Train RMSE", alpha=0.7)
ax.plot(stages, test_rmse_by_stage, label="Test RMSE", alpha=0.7)

# Mark the best test stage
best_stage = stages[np.argmin(test_rmse_by_stage)]
best_rmse = min(test_rmse_by_stage)
ax.axvline(best_stage, color="gray", linestyle="--", alpha=0.5, label=f"Best test: stage {best_stage}")
ax.plot(best_stage, best_rmse, "ko", markersize=8)

ax.set_xlabel("Boosting Stage (number of trees)")
ax.set_ylabel("RMSE")
ax.set_title("Gradient Boosting Learning Curve")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best test RMSE: {best_rmse:.4f} at stage {best_stage}")
print(f"Final test RMSE: {test_rmse_by_stage[-1]:.4f} at stage {len(test_rmse_by_stage)}")

The training RMSE keeps dropping as more trees are added - the model keeps fitting the training data more closely. The test RMSE drops initially (the model is learning real patterns) but eventually levels off or starts to rise (the model starts memorizing noise).

So what's the "right" number of trees? The plot marks the best test stage, but in practice you don't have access to the test set during training. Two common strategies:

- *Hyperparameter search*: include `n_estimators` as a parameter in `RandomizedSearchCV` or `GridSearchCV`. The search evaluates different values (e.g., 100, 200, 300, 500) via cross-validation and picks the best. We'll demonstrate this in Part 3 with XGBoost.
- *Early stopping*: set `n_estimators` to a large value and monitor validation loss during training. Stop when performance hasn't improved for a set number of rounds. XGBoost supports this natively. We'll demonstrate this in Part 3 as well.

The learning rate matters here too. Our model also uses `subsample=0.8`, which means each tree is trained on a random 80% of the training data (the boosting equivalent of bootstrap sampling in RF - it adds noise that acts as regularization). Combined with `max_depth=4`, these settings provide enough regularization that overfitting is gradual. With more aggressive settings (higher learning rate, deeper trees, `subsample=1.0`), the test curve would diverge from train much sooner.

### Learning rate and n_estimators trade off

Lower learning rates generally produce better models but need more trees. Let's see this directly.

In [ ]:
configs = [
    (0.3,  100,  "High lr, few trees"),
    (0.1,  300,  "Medium lr, medium trees"),
    (0.05, 600,  "Low lr, many trees"),
    (0.01, 1000, "Very low lr, lots of trees"),
]

fig, ax = plt.subplots(figsize=(10, 5))

for lr, n_est, label in configs:
    gb_temp = GradientBoostingRegressor(
        n_estimators=n_est, learning_rate=lr, max_depth=4,
        subsample=0.8, random_state=42,
    )
    gb_temp.fit(X_train, y_train)

    test_rmse = [
        np.sqrt(mean_squared_error(y_test, y_pred))
        for y_pred in gb_temp.staged_predict(X_test)
    ]
    ax.plot(range(1, n_est + 1), test_rmse, label=f"lr={lr} ({label})")

ax.set_xlabel("Boosting Stage")
ax.set_ylabel("Test RMSE")
ax.set_title("Learning Rate vs. Number of Estimators")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Lower learning rates reach lower error floors but take more stages to get there. The practical implication: start with `learning_rate=0.1` and `n_estimators=300` as a baseline. If you have time for hyperparameter tuning, try lower learning rates with more trees.

### Feature importance

Like random forests, gradient boosting tracks feature importance. The default `feature_importances_` attribute uses MDI (mean decrease in impurity), same as RF. And like RF, permutation importance is the more reliable measure.

In [ ]:
# MDI importance (built-in)
mdi_imp = gb.feature_importances_

# Permutation importance (model-agnostic)
perm_result = permutation_importance(gb, X_test, y_test, n_repeats=10, random_state=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sorted_mdi = np.argsort(mdi_imp)
ax1.barh(np.array(california.feature_names)[sorted_mdi], mdi_imp[sorted_mdi])
ax1.set_xlabel("MDI Importance")
ax1.set_title("Gradient Boosting — MDI")

sorted_perm = np.argsort(perm_result.importances_mean)
ax2.barh(
    np.array(california.feature_names)[sorted_perm],
    perm_result.importances_mean[sorted_perm],
)
ax2.set_xlabel("Permutation Importance")
ax2.set_title("Gradient Boosting — Permutation")

plt.tight_layout()
plt.show()

Compare these rankings with the RF importance plots from 13a. The top features (`MedInc`, geographic coordinates) should be familiar, but the *relative* rankings may shift. In particular, notice how geographic features (Latitude, Longitude) tend to rank higher for gradient boosting's permutation importance than they did for RF. Why?

Boosting builds trees sequentially, with each tree targeting the residuals of the previous ensemble. Once early trees capture the income signal, later trees focus on what's left - and spatial patterns (neighborhoods, proximity to coast) are a major source of remaining structure. RF trees are independent and each tries to capture everything at once, so they rely more heavily on the single strongest predictor.

### Built-in feature selection

One underappreciated advantage of tree-based models is that they perform feature selection automatically. When a tree chooses which feature to split on, it's implicitly ignoring features that don't help. Irrelevant features simply never get selected for splits - they don't degrade performance the way they can with linear models or KNN.

This matters in practice. With Ridge regression, irrelevant features add noise to the coefficient estimates even after regularization shrinks them toward zero. With KNN, irrelevant features inflate distances in feature space, making neighbors less meaningful (the "curse of dimensionality"). Tree-based models are largely immune to both problems. You can throw 50 features at a random forest or gradient boosting model, and the algorithm will figure out which ones matter - without you having to pre-select them.

This isn't a license to skip feature engineering entirely. Thoughtful features still help, and understanding *which* features matter (via the importance plots above) is essential for interpretation. But the robustness to irrelevant features is a major practical advantage, especially early in an analysis when you're still exploring your data.

---

## Part 3: XGBoost

### The production-grade option

sklearn's `GradientBoostingRegressor` is good for learning but has limitations at scale: it's single-threaded and relatively slow. For production work and competitions, XGBoost (eXtreme Gradient Boosting) is the standard choice. It builds on the same sequential boosting idea but adds:

- Built-in L1 and L2 regularization on the leaf weights
- Automatic handling of missing values
- Parallelized tree construction (not the boosting sequence itself, but the split search within each tree)
- Efficient memory usage for large datasets

XGBoost dominates structured/tabular data tasks so thoroughly that "try XGBoost" has become a default strategy. It's the anti-pattern for the "no free lunch" theorem - it wins so often on tabular data that it's a reasonable default starting point for almost any structured dataset.

A practical note on missing values: sklearn's `GradientBoostingRegressor` cannot handle `NaN` values - you must impute them before fitting. XGBoost (and `HistGradientBoostingRegressor`) handle them natively. During training, when XGBoost encounters a missing value at a split, it tries sending it both left and right and learns which direction reduces the loss more. The optimal direction is stored in the tree, so at prediction time missing values are automatically routed to the better side. This is especially valuable with real-world datasets where missingness is common and informative.

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
xgb.fit(X_train, y_train)

xgb_train_rmse = np.sqrt(mean_squared_error(y_train, xgb.predict(X_train)))
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, xgb.predict(X_test)))

print(f"XGBoost (300 trees, lr=0.1, depth=4):")
print(f"  Train RMSE: {xgb_train_rmse:.4f}")
print(f"  Test RMSE:  {xgb_test_rmse:.4f}")

XGBoost uses the familiar sklearn API - `fit`, `predict`, `score` all work the same way. The key additional hyperparameters beyond what sklearn GBM offers:

- `colsample_bytree`: fraction of features sampled per tree (similar to RF's `max_features`)
- `gamma`: minimum loss reduction required to make a further split (acts as a pruning threshold)
- `reg_alpha`: L1 regularization on leaf weights (encourages sparsity)
- `reg_lambda`: L2 regularization on leaf weights (encourages smaller weights)

### Tuning XGBoost with RandomizedSearchCV

XGBoost has a large hyperparameter space. With 8-9 parameters, each with 3-5 reasonable values, the full grid can easily exceed 100,000 combinations. `RandomizedSearchCV` (which you first saw in the Day 20 workshop) is the practical approach - sample a manageable number of random combinations and find a good-enough configuration.

In [ ]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.1, 0.5, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
}

# Total grid: 4 * 4 * 4 * 4 * 4 * 4 * 4 = 16,384 combinations
# With 5-fold CV, exhaustive search = 81,920 model fits
# RandomizedSearchCV with 50 iterations = 250 model fits

search = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    param_distributions=param_distributions,
    n_iter=50,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
)

start = time.time()
search.fit(X_train, y_train)
elapsed = time.time() - start

print(f"Search completed in {elapsed:.1f}s ({50 * 5} model fits)")
print(f"Best CV RMSE:  {-search.best_score_:.4f}")
print(f"Best params:   {search.best_params_}")

In [ ]:
# Evaluate the tuned model on the held-out test set
tuned_xgb = search.best_estimator_
tuned_test_rmse = np.sqrt(mean_squared_error(y_test, tuned_xgb.predict(X_test)))

print(f"Default XGBoost test RMSE: {xgb_test_rmse:.4f}")
print(f"Tuned XGBoost test RMSE:   {tuned_test_rmse:.4f}")
print(f"Improvement:               {xgb_test_rmse - tuned_test_rmse:.4f}")

The tuned model should improve on the defaults. The size of the improvement depends on how well the defaults happened to fit this dataset. Sometimes the defaults are already close to optimal; sometimes tuning makes a substantial difference. Either way, this is the workflow for production: set up a reasonable search space and let `RandomizedSearchCV` explore it.

### About `RandomizedSearchCV`

How does `RandomizedSearchCV` actually work? It's important to understand: there is no intelligence here. Each of the 50 trials draws a completely random combination of hyperparameters, evaluates it with cross-validation, and records the score. Trial 50 knows nothing about what happened in trials 1-49. There's no gradient, no steering, no learning from past results. It's pure random sampling.

This seems like it shouldn't work, but it does - and the reason is counterintuitive. Bergstra & Bengio (2012) showed that in most hyperparameter spaces, only 2-3 parameters strongly affect performance. The rest are nearly irrelevant. Grid search wastes most of its budget exhaustively varying unimportant parameters while only testing a few values of the important ones. Consider a 7-dimensional grid with 50 total points: that's roughly 2 unique values per dimension. Random search with 50 points, by contrast, produces 50 unique values along *every* dimension - dramatically better coverage of whichever dimensions turn out to matter.

Here's the intuition visually. Imagine two hyperparameters, but only one of them matters (the x-axis). A 5x5 grid tests only 5 unique values of the important parameter, wasting 20 of its 25 trials on variations of the unimportant one. 25 random points give 25 unique values along the important axis.

In [ ]:
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Grid search: 5x5
g = np.linspace(0.1, 0.9, 5)
gx, gy = np.meshgrid(g, g)
axes[0].scatter(gx.ravel(), gy.ravel(), s=50, color="steelblue", edgecolor="black", zorder=3)
# Project onto x-axis
for val in g:
    axes[0].axvline(val, color="steelblue", alpha=0.15, linewidth=6)
axes[0].set_title("Grid Search (25 points, 5 unique x-values)")
axes[0].set_xlabel("Important parameter")
axes[0].set_ylabel("Unimportant parameter")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

# Random search: 25 points
rx = rng.uniform(0.05, 0.95, 25)
ry = rng.uniform(0.05, 0.95, 25)
axes[1].scatter(rx, ry, s=50, color="tomato", edgecolor="black", zorder=3)
for val in rx:
    axes[1].axvline(val, color="tomato", alpha=0.15, linewidth=2)
axes[1].set_title("Random Search (25 points, 25 unique x-values)")
axes[1].set_xlabel("Important parameter")
axes[1].set_ylabel("Unimportant parameter")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

The shaded bands show coverage along the important axis. Grid search (left) tests only 5 values no matter how many total points it uses. Random search (right) tests 25 distinct values with the same budget. The grid's neatness is actually a liability - it guarantees uniform coverage of the *full space* but sacrifices resolution along any single axis. Randomness trades that spatial uniformity for better marginal coverage along every dimension independently. When you don't know in advance which parameters matter most - which is the usual case - random search hedges better.

If you *do* want a search that learns from its own results - using past scores to steer toward promising regions of the hyperparameter space - that's *Bayesian optimization*, implemented in tools like Optuna and scikit-optimize. These model the objective function and choose each next trial to maximize expected improvement. More sample-efficient than random search, but more complex to set up and not always worth the added machinery.

How many random iterations do you need? More is better, but with diminishing returns. Let's see how the best score improves as we increase the number of random trials:

In [ ]:
# Track how best score improves with more iterations
results = pd.DataFrame(search.cv_results_)
best_so_far = []
running_best = np.inf  # best RMSE so far (lower is better)
for score in results["mean_test_score"].values:
    rmse = -score  # convert neg RMSE to positive RMSE
    running_best = min(running_best, rmse)
    best_so_far.append(running_best)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(best_so_far) + 1), best_so_far, linewidth=2)
ax.set_xlabel("Number of Random Trials")
ax.set_ylabel("Best CV RMSE So Far")
ax.set_title("Diminishing Returns of Random Search")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"After 10 trials:  {best_so_far[9]:.4f}")
print(f"After 25 trials:  {best_so_far[24]:.4f}")
print(f"After 50 trials:  {best_so_far[49]:.4f}")

The curve flattens quickly - most of the improvement comes in the first 20-30 trials. In practice, 50-100 iterations is usually enough. If time allows, you can always increase `n_iter`, but the gains diminish.

What does the search space look like? We can visualize the CV scores across two of the most impactful hyperparameters - `learning_rate` and `max_depth` - to see how they interact.

In [ ]:
# Extract results from the search
results["mean_rmse"] = -results["mean_test_score"]

# Pivot on learning_rate vs max_depth (average across other params)
heatmap_data = results.groupby(
    ["param_learning_rate", "param_max_depth"]
)["mean_rmse"].mean().unstack()

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heatmap_data.values, cmap="RdYlGn_r", aspect="auto")

ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)
ax.set_xlabel("max_depth")
ax.set_ylabel("learning_rate")
ax.set_title("Mean CV RMSE by Learning Rate and Max Depth")

# Annotate cells
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        val = heatmap_data.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=10)

fig.colorbar(im, ax=ax, label="CV RMSE")
plt.tight_layout()
plt.show()

Not every cell in the grid was sampled - `RandomizedSearchCV` picks random combinations, so some `(learning_rate, max_depth)` pairs may have many trials while others have few or none. The heatmap shows the average across whatever other parameters were sampled for each pair. The pattern you should see: very low learning rates with shallow trees underperform (not enough capacity), while moderate learning rates with moderate depth hit the sweet spot.

This heatmap is also a tool for iterating on your search space. If the best results cluster near the edge of your grid - say all the best scores are at `learning_rate=0.2`, the highest value you tried - that's a signal to expand the range and try higher values (0.3, 0.5). If a whole region is uniformly poor (like `learning_rate=0.01` here), you can narrow the grid to focus on the promising region. In practice, hyperparameter tuning is often a two-pass process: a coarse random search to identify the promising region, then a finer search within that region.

### Early stopping

Early stopping is a general regularization technique: train an iterative model, monitor performance on held-out data, and stop when it stops improving. The idea applies anywhere training happens in stages - neural networks stop after a certain number of epochs, and iterative solvers like `LogisticRegression(solver='saga')` can stop after a set number of iterations. For gradient boosting, it's especially natural because each tree is a discrete step and the learning curve (Part 2) shows exactly when more trees stop helping.

XGBoost makes early stopping easy. Set `n_estimators` to a large ceiling and specify `early_stopping_rounds` - if the validation score doesn't improve for that many consecutive rounds, training stops automatically.

In [ ]:
# Split off a validation set for early stopping
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

xgb_early = XGBRegressor(
    n_estimators=2000,        # set high - early stopping will cut it short
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=20, # stop if no improvement for 20 rounds
    random_state=42,
)

xgb_early.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

early_rmse = np.sqrt(mean_squared_error(y_test, xgb_early.predict(X_test)))
print(f"Early stopping triggered at round: {xgb_early.best_iteration}")
print(f"Test RMSE: {early_rmse:.4f}")
print(f"Trees used: {xgb_early.best_iteration} / 2000")

Notice we split off a separate validation set here rather than using cross-validation. In earlier lectures, we favored CV because it uses all the data for both training and evaluation. Early stopping is different - it needs to check validation loss *during* training, at every boosting round. Running full CV at every round would be prohibitively expensive. The trade-off is worth it: we sacrifice 20% of our training data for validation, but in exchange we automatically find the optimal number of trees without any manual tuning or exhaustive search.

Early stopping is a very practical approach in general. Set `n_estimators` to something large (1000-2000), pick a reasonable learning rate (0.05-0.1), and let early stopping find the right number of trees automatically. This is faster than searching over `n_estimators` in a grid and often produces better results because it adapts to the specific dataset.

### The gradient boosting landscape

XGBoost isn't the only production-grade gradient boosting library. Two other major options:

- LightGBM (Microsoft) is faster than XGBoost on large datasets, handles categorical features natively, and is preferred when training speed is a concern. Algorithmic difference: it grows trees leaf-wise (best-first) rather than level-wise, which often reaches lower error with fewer leaves.

- CatBoost (Yandex) has the best native handling of categorical features and is particularly effective when your data has many categorical columns. It uses ordered boosting to reduce prediction shift.

- sklearn's `HistGradientBoostingRegressor` uses the same histogram-based binning approach as LightGBM. It handles missing values natively, supports categorical features, and is dramatically faster than sklearn's original `GradientBoostingRegressor` on datasets with more than ~10,000 samples. It's the right choice when you want fast gradient boosting without leaving the sklearn ecosystem.

All four produce comparable accuracy on most datasets. The choice typically comes down to speed, categorical feature handling, and ecosystem preference. XGBoost is the recommended default for most tabular data work - it has the largest community, the most tutorials, and the most robust documentation.

Let's compare the three options we have access to - sklearn's original GBR, HistGradientBoosting, and XGBoost - on the same dataset.

In [ ]:
# Same-ish hyperparameters across all three for a fair comparison
gb_compare = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.1, max_depth=4, random_state=42,
)
hgb = HistGradientBoostingRegressor(
    max_iter=300, learning_rate=0.1, max_depth=4, random_state=42,
)
xgb_compare = XGBRegressor(
    n_estimators=300, learning_rate=0.1, max_depth=4, random_state=42,
)

compare_models = {
    "GradientBoostingRegressor": gb_compare,
    "HistGradientBoostingRegressor": hgb,
    "XGBRegressor": xgb_compare,
}

compare_rows = []
for name, model in compare_models.items():
    start = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    compare_rows.append({
        "Model": name,
        "Test RMSE": f"{test_rmse:.4f}",
        "Fit time (s)": f"{fit_time:.2f}",
    })

compare_df = pd.DataFrame(compare_rows)
print(compare_df.to_string(index=False))

The accuracy is comparable across all three, but the speed difference is dramatic. `HistGradientBoostingRegressor` and XGBoost both use histogram-based binning, which explains their similar speed. sklearn's original `GradientBoostingRegressor` evaluates every unique feature value at every split - much slower, but useful for understanding the algorithm. For datasets larger than ~10,000 samples, prefer HGBR or XGBoost.

---

## Part 4: Baseline Hierarchy

We've been building a progression of models across this course. Each new entry tells a story about the data. Let's add gradient boosting and see where it lands.

In [ ]:
models = {
    "Dummy (mean)": DummyRegressor(),
    "Ridge": Ridge(),
    "KNN (scaled)": Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor())]),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.1, max_depth=4, random_state=42,
    ),
    "XGBoost (tuned)": search.best_estimator_,
}

rows = []
for name, model in models.items():
    start = time.time()
    cv = cross_val_score(
        model, X_train, y_train, cv=5,
        scoring="neg_root_mean_squared_error",
    )
    elapsed = time.time() - start
    rows.append({
        "Model": name,
        "CV RMSE (mean)": f"{-cv.mean():.4f}",
        "CV RMSE (std)": f"{cv.std():.4f}",
        "Time (s)": f"{elapsed:.1f}",
    })

hierarchy = pd.DataFrame(rows)
print(hierarchy.to_string(index=False))

In [ ]:
# Visualize the full hierarchy
rmse_vals = [float(r["CV RMSE (mean)"]) for r in rows]
model_names = [r["Model"] for r in rows]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(model_names, rmse_vals, color="steelblue", edgecolor="black")
ax.set_xlabel("CV RMSE (lower is better)")
ax.set_title("Baseline Hierarchy — California Housing")
ax.invert_yaxis()

for bar, val in zip(bars, rmse_vals):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

Reading the gaps:

| Gap | What it reveals |
|---|---|
| Dummy → Ridge | Linear signal exists |
| Ridge → KNN | Nonlinearity matters |
| KNN ≈ Decision Tree | Both capture non-linear structure via different mechanisms |
| DT → Random Forest | Variance reduction through averaging |
| RF → Gradient Boosting | Bias reduction through sequential correction |
| GB → XGBoost (tuned) | Regularization + hyperparameter tuning |

The DT → RF gap showed us that averaging helps. The RF → GB gap shows us that sequential error correction helps *on top of that*. These are complementary strategies: RF attacks variance, GB attacks bias. The tuned XGBoost entry shows the additional benefit of regularization and careful tuning.

Notice the time column. Gradient boosting is slower than RF because it builds trees sequentially - each tree must wait for the previous one to finish before computing residuals. RF can parallelize across all cores. This is the fundamental speed/accuracy tradeoff between bagging and boosting.

### When to use what

| | Random Forest | Gradient Boosting |
|---|---|---|
| Training speed | Fast (parallelizable) | Slower (sequential) |
| Overfitting risk | Low - more trees rarely hurts | Higher - too many trees overfits |
| Hyperparameter sensitivity | Forgiving - defaults often good | Sensitive - tuning matters more |
| Scaling required | No | No |
| Best for | Quick baseline, robust default | Maximum accuracy when tuning time available |

A common workflow: start with RF as a quick baseline (it's harder to mess up), then try XGBoost with `RandomizedSearchCV` to see if tuning improves things. The comparison between the two tells a story about whether bias reduction helps on a given dataset.

---

## Summary

| Property | Random Forest | Gradient Boosting | XGBoost |
|---|---|---|---|
| Strategy | Bagging (parallel) | Boosting (sequential) | Boosting + regularization |
| Reduces | Variance | Bias | Bias |
| Tree depth | Deep (full) | Shallow (3-5) | Shallow (3-5) |
| More trees... | Always safe | Can overfit | Can overfit |
| Scaling needed | No | No | No |
| Feature selection | Built-in | Built-in | Built-in |
| Speed | Fast (parallel) | Slow (sequential) | Fast (optimized) |

Key advantages of tree-based ensembles over linear models and KNN:

- No scaling required - splits are threshold-based, invariant to monotonic transforms
- Built-in feature selection - irrelevant features are ignored automatically, no manual filtering needed
- Capture non-linear relationships and feature interactions without explicit feature engineering
- Feature importance rankings come free with the model, aiding interpretation

This notebook used regression throughout for continuity with 12b and 13a, but everything applies to classification too. The API swap is the same pattern you've seen with decision trees: `GradientBoostingClassifier`, `HistGradientBoostingClassifier`, `XGBClassifier`. Same hyperparameters, same tuning workflow - just swap the estimator and scoring metric.

The complete baseline hierarchy:

Dummy → Ridge → KNN → Decision Tree → Random Forest → Gradient Boosting/XGBoost

Each gap in this hierarchy reveals something about your data. The goal isn't just to find the best model - it's to understand *why* each step improves (or doesn't improve) over the last. That diagnostic reasoning is what separates ML engineering from trial and error.